# Polymer Matrix Multiplier Supervisor

Canonical unified polymer notebook. Edit the config cell below for one-off runs, or change `systems/polymer/notebook_params.py` for repo-wide defaults.

## User Config

In [1]:
from pathlib import Path
import os

from systems.polymer import get_polymer_notebook_defaults
from systems.polymer.data_io import canonical_baseline_path
from utils.notebook_setup import prepare_polymer_notebook_env, print_grouped_notebook_summary

NB = get_polymer_notebook_defaults("matrix")

# Main notebook controls.
AGENT_KIND = NB["agent_kind"]  # "td3" | "sac"
RUN_MODE = NB["run_mode"]  # "nominal" | "disturb"
STATE_MODE = NB["state_mode"]  # "standard" | "mismatch"
STYLE_PROFILE = NB["style_profile"]  # "hybrid" | "paper" | "debug"
SAVE_PDF = NB["save_pdf"]

POLYMER_DATA_DIR_OVERRIDE = NB["data_dir_override"]
POLYMER_RESULTS_DIR_OVERRIDE = NB["results_dir_override"]
RESULT_PREFIX_OVERRIDE = NB["result_prefix_override"]
COMPARE_PREFIX_OVERRIDE = NB["compare_prefix_override"]
BASELINE_MPC_PATH_OVERRIDE = NB["baseline_mpc_path_override"]
N_TESTS_OVERRIDE = NB["n_tests_override"]
SET_POINTS_LEN_OVERRIDE = NB["set_points_len_override"]
WARM_START_OVERRIDE = NB["warm_start_override"]
TEST_CYCLE_OVERRIDE = NB["test_cycle_override"]
PLOT_START_EPISODE_OVERRIDE = NB["plot_start_episode_override"]
COMPARE_START_EPISODE_OVERRIDE = NB["compare_start_episode_override"]

REPO_ROOT, DATA_DIR, RESULT_DIR = prepare_polymer_notebook_env(
    data_dir_override=POLYMER_DATA_DIR_OVERRIDE,
    results_dir_override=POLYMER_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)
RUN_PROFILE = NB["run_profiles"][(AGENT_KIND, RUN_MODE)]


## Imports

In [2]:
import numpy as np
import torch

from SACAgent.sac_agent import SACAgent
from Simulation.mpc import MpcSolverGeneral
from Simulation.system_functions import PolymerCSTR
from TD3Agent.agent import TD3Agent
from systems.polymer import (
    POLYMER_DELTA_T_HOURS,
    POLYMER_DESIGN_PARAMS,
    POLYMER_INPUT_BOUNDS,
    POLYMER_OBSERVER_POLES,
    POLYMER_RL_SETPOINTS_PHYS,
    POLYMER_SETPOINT_RANGE_PHYS,
    POLYMER_SYSTEM_METADATA,
    POLYMER_SS_INPUTS,
    POLYMER_SYSTEM_PARAMS,
    RL_REWARD_DEFAULTS,
    load_polymer_system_data,
)
from utils.helpers import apply_min_max
from utils.matrix_runner import run_matrix_multiplier_supervisor
from utils.multiplier_sensitivity import (
    extract_suggested_bounds_from_diagnostic,
    format_multiplier_sensitivity_summary,
    run_scalar_matrix_sensitivity,
    save_multiplier_sensitivity_outputs,
    timestamped_sensitivity_output_dir,
)
from utils.observer import compute_observer_gain
from utils.plotting import compare_mpc_rl_from_dirs, plot_matrix_multiplier_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim

## System And Data Setup

In [3]:
# Build the plant, load the canonical data bundle, and prepare the supervisory setpoint scenario.
SYS = NB["system_setup"]
system_params = SYS["system_params"].copy()
system_design_params = SYS["design_params"].copy()
system_steady_state_inputs = SYS["ss_inputs"].copy()
delta_t = SYS["delta_t_hours"]

cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states = {"ss_inputs": cstr_ss.ss_inputs, "y_ss": cstr_ss.y_ss}

setpoint_y = SYS["setpoint_range_phys"].copy()
u_min = SYS["input_bounds"]["u_min"].copy()
u_max = SYS["input_bounds"]["u_max"].copy()
system_data = load_polymer_system_data(
    REPO_ROOT,
    steady_states=steady_states,
    setpoint_y=setpoint_y,
    u_min=u_min,
    u_max=u_max,
    n_inputs=2,
    data_override=POLYMER_DATA_DIR_OVERRIDE,
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]

inputs_number = int(B_aug.shape[1])
y_sp_scenario_phys = SYS["rl_setpoints_phys"].copy()
y_sp_scenario = (
    apply_min_max(y_sp_scenario_phys, data_min[inputs_number:], data_max[inputs_number:])
    - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])
)


## Run / Reward / Agent Setup

In [4]:
# Run-profile, controller, reward, and agent setup.
EPISODE_CFG = NB["episode_defaults"]
CTRL = NB["controller"]
TD3_CFG = NB["td3_agent"]
SAC_CFG = NB["sac_agent"]
REWARD_CFG = NB["reward"]
BEHAVIORAL_CLONING = dict(NB.get("behavioral_cloning", {}))
OFFLINE_MULTIPLIER_DIAGNOSTICS = dict(CTRL.get("offline_multiplier_diagnostics", {}))
RELEASE_PROTECTED_ADVISORY_CAPS = dict(CTRL.get("release_protected_advisory_caps", {}))
MPC_ACCEPTANCE_FALLBACK = dict(CTRL.get("mpc_acceptance_fallback", {}))
if bool(BEHAVIORAL_CLONING.get("enabled", False)) and bool(RELEASE_PROTECTED_ADVISORY_CAPS.get("enabled", False)) and not bool(MPC_ACCEPTANCE_FALLBACK.get("enabled", False)):
    HANDOFF_METHOD = "Step 4G (BC + release guard)"
elif bool(BEHAVIORAL_CLONING.get("enabled", False)) and not bool(RELEASE_PROTECTED_ADVISORY_CAPS.get("enabled", False)):
    HANDOFF_METHOD = "Step 4 (BC only)"
elif bool(RELEASE_PROTECTED_ADVISORY_CAPS.get("enabled", False)) and not bool(BEHAVIORAL_CLONING.get("enabled", False)):
    HANDOFF_METHOD = "Step 2 (release guard only)"
else:
    HANDOFF_METHOD = "Custom / mixed handoff"

n_tests = int(EPISODE_CFG["n_tests"] if N_TESTS_OVERRIDE is None else N_TESTS_OVERRIDE)
set_points_len = int(EPISODE_CFG["set_points_len"] if SET_POINTS_LEN_OVERRIDE is None else SET_POINTS_LEN_OVERRIDE)
warm_start = int(EPISODE_CFG["warm_start"] if WARM_START_OVERRIDE is None else WARM_START_OVERRIDE)
TEST_CYCLE = list(EPISODE_CFG["test_cycle"] if TEST_CYCLE_OVERRIDE is None else TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = int(RUN_PROFILE["plot_start_episode"] if PLOT_START_EPISODE_OVERRIDE is None else PLOT_START_EPISODE_OVERRIDE)
COMPARE_START_EPISODE = int(RUN_PROFILE["compare_start_episode"] if COMPARE_START_EPISODE_OVERRIDE is None else COMPARE_START_EPISODE_OVERRIDE)
RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or RUN_PROFILE["result_prefix"]
COMPARE_PREFIX = COMPARE_PREFIX_OVERRIDE or RUN_PROFILE["compare_prefix"]
BASELINE_MPC_PATH = Path(BASELINE_MPC_PATH_OVERRIDE).expanduser() if BASELINE_MPC_PATH_OVERRIDE else canonical_baseline_path(REPO_ROOT, RUN_MODE, data_override=POLYMER_DATA_DIR_OVERRIDE)

N_INPUTS = int(B_aug.shape[1])
N_OUTPUTS = int(C_aug.shape[0])
STATE_DIM = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, STATE_MODE)
MISMATCH_CLIP = CTRL["mismatch_clip"]
INNOVATION_SCALE_MODE = CTRL["innovation_scale_mode"]
INNOVATION_SCALE_REF = CTRL["innovation_scale_ref"]
TRACKING_SCALE_MODE = CTRL["tracking_scale_mode"]
TRACKING_ETA_TOL = CTRL["tracking_eta_tol"]
TRACKING_SCALE_FLOOR = CTRL["tracking_scale_floor"]
TRACKING_SCALE_FLOOR_MODE = CTRL["tracking_scale_floor_mode"]
BASE_STATE_NORM_MODE = CTRL["base_state_norm_mode"]
BASE_STATE_RUNNING_NORM_CLIP = CTRL["base_state_running_norm_clip"]
BASE_STATE_RUNNING_NORM_EPS = CTRL["base_state_running_norm_eps"]
MISMATCH_FEATURE_TRANSFORM_MODE = CTRL["mismatch_feature_transform_mode"]
MISMATCH_TRANSFORM_TANH_SCALE = CTRL["mismatch_transform_tanh_scale"]
MISMATCH_TRANSFORM_POST_CLIP = CTRL["mismatch_transform_post_clip"]
OBSERVER_UPDATE_ALIGNMENT = CTRL["observer_update_alignment"]
ACTION_DIM = 1 + N_INPUTS
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

predict_h = CTRL["predict_h"]
cont_h = CTRL["cont_h"]
Q1_penalty = CTRL["Q1_penalty"]
Q2_penalty = CTRL["Q2_penalty"]
R1_penalty = CTRL["R1_penalty"]
R2_penalty = CTRL["R2_penalty"]
LOW_COEF = CTRL["low_coef"].copy()
HIGH_COEF = CTRL["high_coef"].copy()
TD3_EXPLORATION_MODE = TD3_CFG["exploration_mode"]
TD3_N_STEP = int(TD3_CFG["n_step"])
TD3_MULTISTEP_MODE = TD3_CFG["multistep_mode"]
TD3_LAMBDA_VALUE = float(TD3_CFG["lambda_value"])
TD3_LOSS_TYPE = TD3_CFG["loss_type"]
TD3_PARAM_NOISE_RESAMPLE_INTERVAL = TD3_CFG["param_noise_resample_interval"]
TD3_BUFFER_SIZE = int(TD3_CFG["buffer_size"])
TD3_REPLAY_FRAC_PER = float(TD3_CFG["replay_frac_per"])
TD3_REPLAY_FRAC_RECENT = float(TD3_CFG["replay_frac_recent"])
TD3_REPLAY_RECENT_WINDOW_MULT = int(TD3_CFG["replay_recent_window_mult"])
TD3_REPLAY_RECENT_WINDOW = int(TD3_CFG["replay_recent_window"]) if TD3_CFG["replay_recent_window"] is not None else min(TD3_BUFFER_SIZE, TD3_REPLAY_RECENT_WINDOW_MULT * set_points_len)
TD3_REPLAY_ALPHA = float(TD3_CFG["replay_alpha"])
TD3_REPLAY_BETA_START = float(TD3_CFG["replay_beta_start"])
TD3_REPLAY_BETA_END = float(TD3_CFG["replay_beta_end"])
TD3_REPLAY_BETA_STEPS = int(TD3_CFG["replay_beta_steps"])
SAC_BUFFER_SIZE = int(SAC_CFG["buffer_size"])
SAC_REPLAY_FRAC_PER = float(SAC_CFG["replay_frac_per"])
SAC_REPLAY_FRAC_RECENT = float(SAC_CFG["replay_frac_recent"])
SAC_REPLAY_RECENT_WINDOW_MULT = int(SAC_CFG["replay_recent_window_mult"])
SAC_REPLAY_RECENT_WINDOW = int(SAC_CFG["replay_recent_window"]) if SAC_CFG["replay_recent_window"] is not None else min(SAC_BUFFER_SIZE, SAC_REPLAY_RECENT_WINDOW_MULT * set_points_len)
SAC_REPLAY_ALPHA = float(SAC_CFG["replay_alpha"])
SAC_REPLAY_BETA_START = float(SAC_CFG["replay_beta_start"])
SAC_REPLAY_BETA_END = float(SAC_CFG["replay_beta_end"])
SAC_REPLAY_BETA_STEPS = int(SAC_CFG["replay_beta_steps"])
SAC_LOSS_TYPE = SAC_CFG["loss_type"]
SAC_N_STEP = int(SAC_CFG["n_step"])
SAC_MULTISTEP_MODE = SAC_CFG["multistep_mode"]
SAC_LAMBDA_VALUE = float(SAC_CFG["lambda_value"])
N_STEP = TD3_N_STEP if AGENT_KIND == "td3" else SAC_N_STEP
MULTISTEP_MODE = TD3_MULTISTEP_MODE if AGENT_KIND == "td3" else SAC_MULTISTEP_MODE
LAMBDA_VALUE = TD3_LAMBDA_VALUE if AGENT_KIND == "td3" else SAC_LAMBDA_VALUE
USE_SHIFTED_MPC_WARM_START = CTRL["use_shifted_mpc_warm_start"]
ACTIVE_BUFFER_SIZE = TD3_BUFFER_SIZE if AGENT_KIND == "td3" else SAC_BUFFER_SIZE
ACTIVE_REPLAY_SETTINGS = {
    "buffer_size": ACTIVE_BUFFER_SIZE,
    "replay_frac_per": TD3_REPLAY_FRAC_PER if AGENT_KIND == "td3" else SAC_REPLAY_FRAC_PER,
    "replay_frac_recent": TD3_REPLAY_FRAC_RECENT if AGENT_KIND == "td3" else SAC_REPLAY_FRAC_RECENT,
    "replay_recent_window_mult": TD3_REPLAY_RECENT_WINDOW_MULT if AGENT_KIND == "td3" else SAC_REPLAY_RECENT_WINDOW_MULT,
    "replay_recent_window": TD3_REPLAY_RECENT_WINDOW if AGENT_KIND == "td3" else SAC_REPLAY_RECENT_WINDOW,
    "replay_alpha": TD3_REPLAY_ALPHA if AGENT_KIND == "td3" else SAC_REPLAY_ALPHA,
    "replay_beta_start": TD3_REPLAY_BETA_START if AGENT_KIND == "td3" else SAC_REPLAY_BETA_START,
    "replay_beta_end": TD3_REPLAY_BETA_END if AGENT_KIND == "td3" else SAC_REPLAY_BETA_END,
    "replay_beta_steps": TD3_REPLAY_BETA_STEPS if AGENT_KIND == "td3" else SAC_REPLAY_BETA_STEPS,
}
nominal_qs = CTRL["nominal_qs"]
nominal_qi = CTRL["nominal_qi"]
nominal_hA = CTRL["nominal_ha"]
qi_change = CTRL["qi_change"]
qs_change = CTRL["qs_change"]
ha_change = CTRL["ha_change"]

MPC_obj = MpcSolverGeneral(A_aug, B_aug, C_aug, Q_out=np.array([Q1_penalty, Q2_penalty], float), R_in=np.array([R1_penalty, R2_penalty], float), NP=predict_h, NC=cont_h)
# Reward setup.
reward_params, reward_fn = make_reward_fn_relative_QR(data_min, data_max, N_INPUTS, **REWARD_CFG)

# Agent setup.
if AGENT_KIND == "td3":
    matrix_agent = TD3Agent(
        state_dim=STATE_DIM,
        action_dim=ACTION_DIM,
        actor_hidden=list(TD3_CFG["actor_hidden"]),
        critic_hidden=list(TD3_CFG["critic_hidden"]),
        gamma=TD3_CFG["gamma"],
        actor_lr=TD3_CFG["actor_lr"],
        critic_lr=TD3_CFG["critic_lr"],
        batch_size=TD3_CFG["batch_size"],
        policy_delay=TD3_CFG["policy_delay"],
        target_policy_smoothing_noise_std=TD3_CFG["target_policy_smoothing_noise_std"],
        noise_clip=TD3_CFG["noise_clip"],
        max_action=TD3_CFG["max_action"],
        tau=TD3_CFG["tau"],
        std_start=TD3_CFG["std_start"],
        std_end=TD3_CFG["std_end"],
        std_decay_rate=TD3_CFG["std_decay_rate"],
        std_decay_mode=TD3_CFG["std_decay_mode"],
        buffer_size=TD3_BUFFER_SIZE,
        replay_frac_per=TD3_REPLAY_FRAC_PER,
        replay_frac_recent=TD3_REPLAY_FRAC_RECENT,
        replay_recent_window=TD3_REPLAY_RECENT_WINDOW,
        replay_alpha=TD3_REPLAY_ALPHA,
        replay_beta_start=TD3_REPLAY_BETA_START,
        replay_beta_end=TD3_REPLAY_BETA_END,
        replay_beta_steps=TD3_REPLAY_BETA_STEPS,
        device=DEVICE,
        actor_freeze=TD3_CFG["actor_freeze"],
        exploration_mode=TD3_EXPLORATION_MODE,
        loss_type=TD3_LOSS_TYPE,
        param_noise_resample_interval=TD3_PARAM_NOISE_RESAMPLE_INTERVAL,
        n_step=TD3_N_STEP,
        multistep_mode=TD3_MULTISTEP_MODE,
        lambda_value=TD3_LAMBDA_VALUE,
    )
elif AGENT_KIND == "sac":
    target_entropy = -ACTION_DIM if SAC_CFG["target_entropy"] == "auto_negative_action_dim" else SAC_CFG["target_entropy"]
    matrix_agent = SACAgent(
        state_dim=STATE_DIM,
        action_dim=ACTION_DIM,
        actor_hidden=list(SAC_CFG["actor_hidden"]),
        critic_hidden=list(SAC_CFG["critic_hidden"]),
        gamma=SAC_CFG["gamma"],
        actor_lr=SAC_CFG["actor_lr"],
        critic_lr=SAC_CFG["critic_lr"],
        alpha_lr=SAC_CFG["alpha_lr"],
        batch_size=SAC_CFG["batch_size"],
        grad_clip_norm=SAC_CFG["grad_clip_norm"],
        init_alpha=SAC_CFG["init_alpha"],
        learn_alpha=SAC_CFG["learn_alpha"],
        target_entropy=target_entropy,
        target_update=SAC_CFG["target_update"],
        tau=SAC_CFG["tau"],
        hard_update_interval=SAC_CFG["hard_update_interval"],
        activation=SAC_CFG["activation"],
        use_layernorm=SAC_CFG["use_layernorm"],
        dropout=SAC_CFG["dropout"],
        max_action=SAC_CFG["max_action"],
        buffer_size=SAC_BUFFER_SIZE,
        replay_frac_per=SAC_REPLAY_FRAC_PER,
        replay_frac_recent=SAC_REPLAY_FRAC_RECENT,
        replay_recent_window=SAC_REPLAY_RECENT_WINDOW,
        replay_alpha=SAC_REPLAY_ALPHA,
        replay_beta_start=SAC_REPLAY_BETA_START,
        replay_beta_end=SAC_REPLAY_BETA_END,
        replay_beta_steps=SAC_REPLAY_BETA_STEPS,
        device=DEVICE,
        use_adamw=SAC_CFG["use_adamw"],
        actor_freeze=SAC_CFG["actor_freeze"],
        loss_type=SAC_LOSS_TYPE,
        n_step=SAC_N_STEP,
        multistep_mode=SAC_MULTISTEP_MODE,
        lambda_value=SAC_LAMBDA_VALUE,
    )
else:
    raise ValueError("AGENT_KIND must be 'td3' or 'sac'.")

REPLAY_SETTINGS = ACTIVE_REPLAY_SETTINGS


## Resolved Summary

In [5]:
print_grouped_notebook_summary(
    "Polymer Matrix Multiplier Supervisor run summary",
    {
        "Paths": {"Repo root": REPO_ROOT, "Data dir": DATA_DIR, "Results dir": RESULT_DIR, "Baseline MPC": BASELINE_MPC_PATH},
        "Run setup": {"Agent kind": AGENT_KIND, "Run mode": RUN_MODE, "State mode": STATE_MODE, "n_tests": n_tests, "set_points_len": set_points_len, "warm_start": warm_start, "test_cycle": TEST_CYCLE, "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START, "estimator_mode": "fixed_nominal"},
        "System / controller": {"delta_t_hours": delta_t, "predict_h": predict_h, "cont_h": cont_h, "Q penalties": [Q1_penalty, Q2_penalty], "R penalties": [R1_penalty, R2_penalty], "observer_poles": POLYMER_OBSERVER_POLES.tolist()},
        "Reward": reward_params,
        "Handoff": {"method": HANDOFF_METHOD, "offline_diagnostic_enabled": bool(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("enabled", False)), "action_freeze_subepisodes": int(NB["post_warm_start_action_freeze_subepisodes"]), "actor_freeze_subepisodes": int(NB["post_warm_start_actor_freeze_subepisodes"])},
        "Behavioral cloning": BEHAVIORAL_CLONING,
        "Release guard": RELEASE_PROTECTED_ADVISORY_CAPS,
        "Acceptance / fallback": MPC_ACCEPTANCE_FALLBACK,
        "Agent": {"supervisor": "matrix multiplier", "buffer_size": (TD3_CFG if AGENT_KIND == "td3" else SAC_CFG)["buffer_size"], "n_step": N_STEP, "multistep_mode": MULTISTEP_MODE, "lambda_value": LAMBDA_VALUE, "exploration_mode": TD3_EXPLORATION_MODE if AGENT_KIND == "td3" else "policy_stochastic", "loss_type": TD3_LOSS_TYPE if AGENT_KIND == "td3" else SAC_LOSS_TYPE},
        "Replay": REPLAY_SETTINGS,
        "Mismatch": {"clip": MISMATCH_CLIP, "innovation_scale_mode": INNOVATION_SCALE_MODE, "tracking_scale_mode": TRACKING_SCALE_MODE, "tracking_eta_tol": TRACKING_ETA_TOL, "tracking_scale_floor_mode": TRACKING_SCALE_FLOOR_MODE},
        "Plotting / export": {"style_profile": STYLE_PROFILE, "save_pdf": SAVE_PDF, "result_prefix": RESULT_PREFIX, "compare_prefix": COMPARE_PREFIX, "plot_start_episode": PLOT_START_EPISODE, "compare_start_episode": COMPARE_START_EPISODE},
    },
)

Polymer Matrix Multiplier Supervisor run summary

[Paths]
  Repo root           : C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer
  Data dir            : C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Polymer\Data
  Results dir         : C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Polymer\Results
  Baseline MPC        : C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Polymer\Data\mpc_results_dist.pickle

[Run setup]
  Agent kind          : td3
  Run mode            : disturb
  State mode          : mismatch
  n_tests             : 200
  set_points_len      : 400
  warm_start          : 10
  test_cycle          : [False, False, False, False, False]
  use_shifted_mpc_warm_start: False
  estimator_mode      : fixed_nominal

[System / controller]
  delta_t_hours       : 0.5
  predict_h           : 9
  cont_h              : 3
  Q penalties         : [5.0, 1.0]
  R penalties         : [1.0, 1.0]
  observer_poles      : [0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966, 0.429006

In [6]:
# Offline rho + finite-horizon gain sensitivity diagnostic.
offline_multiplier_diagnostic_result = None
offline_multiplier_diagnostic_paths = {}

if bool(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("enabled", False)):
    if bool(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("apply_suggested_caps", False)):
        print("Step 1 diagnostic is advisory only; apply_suggested_caps is ignored.")
    offline_multiplier_diagnostic_result = run_scalar_matrix_sensitivity(
        A_aug=A_aug,
        B_aug=B_aug,
        C_aug=C_aug,
        low_bounds=LOW_COEF,
        high_bounds=HIGH_COEF,
        predict_h=predict_h,
        n_outputs=N_OUTPUTS,
        epsilon_log=float(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("epsilon_log", 0.02)),
        n_random_samples=int(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("n_random_samples", 2_000)),
        seed=int(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("seed", 42)),
        rho_target=float(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("rho_target", 0.995)),
        gain_threshold=float(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("gain_threshold", 0.25)),
        system_name="polymer",
        method_family="matrix",
    )
    if bool(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("save_outputs", True)):
        offline_multiplier_diagnostic_dir = timestamped_sensitivity_output_dir(
            RESULT_DIR,
            system_name="polymer",
            method_family="matrix",
            agent_kind=AGENT_KIND,
            run_mode=RUN_MODE,
        )
        offline_multiplier_diagnostic_paths = save_multiplier_sensitivity_outputs(
            offline_multiplier_diagnostic_result,
            offline_multiplier_diagnostic_dir,
            make_plots=bool(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("make_plots", True)),
        )
        print(f"Offline multiplier diagnostic outputs: {offline_multiplier_diagnostic_dir}")
    print(format_multiplier_sensitivity_summary(offline_multiplier_diagnostic_result))
else:
    print("Offline multiplier sensitivity diagnostic disabled for this notebook.")

# Step 2 release-protected advisory caps. Diagnostic bounds are used only as an execution guard.
if bool(RELEASE_PROTECTED_ADVISORY_CAPS.get("enabled", False)):
    if offline_multiplier_diagnostic_result is None:
        raise RuntimeError(
            "Step 2 release-protected advisory caps are enabled, but Step 1 offline multiplier diagnostic did not run."
        )
    RELEASE_PROTECTED_ADVISORY_CAPS["advisory_bounds"] = extract_suggested_bounds_from_diagnostic(
        offline_multiplier_diagnostic_result,
        labels=["alpha"] + [f"B_col_{idx + 1}" for idx in range(B_aug.shape[1])],
    )
    print("Step 2 release-protected advisory caps enabled; using Step 1 diagnostic bounds during live release.")
else:
    print("Step 2 release-protected advisory caps disabled for this notebook.")


Offline multiplier diagnostic outputs: C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Polymer\Results\offline_multiplier_sensitivity\polymer_matrix_td3_disturb_20260424_235044
Offline multiplier sensitivity diagnostic
  system: polymer
  method: matrix
  nominal rho: 0.946394
  nominal gain drift: 0
  sampled candidates: 2000
  unstable fraction: 0.0000
  near-unit fraction: 0.0060
  gain ratio p95/max: 0.7344 / 0.7855
  most sensitive coordinates:
    alpha (A_scalar): S_rho=0.9465, S_G=4.157, rho-=0.927654, rho+=0.965512, G-=0.07381, G+=0.08308
    B_col_2 (B_col): S_rho=0, S_G=0.8935, rho-=0.946394, rho+=0.946394, G-=0.01742, G+=0.01777
    B_col_1 (B_col): S_rho=0, S_G=0.4775, rho-=0.946394, rho+=0.946394, G-=0.009413, G+=0.009603
Step 2 release-protected advisory caps enabled; using Step 1 diagnostic bounds during live release.


## Run

In [7]:
# Assemble the shared runner configuration and execute the rollout.
matrix_cfg = {
    "agent_kind": AGENT_KIND,
    "run_mode": RUN_MODE,
    "state_mode": STATE_MODE,
        "mismatch_clip": MISMATCH_CLIP,
    "innovation_scale_mode": INNOVATION_SCALE_MODE,
    "innovation_scale_ref": INNOVATION_SCALE_REF,
    "tracking_scale_mode": TRACKING_SCALE_MODE,
    "tracking_eta_tol": TRACKING_ETA_TOL,
    "tracking_scale_floor": TRACKING_SCALE_FLOOR,
    "tracking_scale_floor_mode": TRACKING_SCALE_FLOOR_MODE,
    "base_state_norm_mode": BASE_STATE_NORM_MODE,
    "base_state_running_norm_clip": BASE_STATE_RUNNING_NORM_CLIP,
    "base_state_running_norm_eps": BASE_STATE_RUNNING_NORM_EPS,
    "mismatch_feature_transform_mode": MISMATCH_FEATURE_TRANSFORM_MODE,
    "mismatch_transform_tanh_scale": MISMATCH_TRANSFORM_TANH_SCALE,
    "mismatch_transform_post_clip": MISMATCH_TRANSFORM_POST_CLIP,
    "observer_update_alignment": OBSERVER_UPDATE_ALIGNMENT,
    "notebook_source": "RL_assisted_MPC_matrices_unified.ipynb",
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "n_step": N_STEP,
    "multistep_mode": MULTISTEP_MODE,
    "lambda_value": LAMBDA_VALUE,
    "warm_start": warm_start,
    "post_warm_start_action_freeze_subepisodes": int(NB["post_warm_start_action_freeze_subepisodes"]),
    "post_warm_start_actor_freeze_subepisodes": int(NB["post_warm_start_actor_freeze_subepisodes"]),
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START,
    "low_coef": LOW_COEF,
    "high_coef": HIGH_COEF,
    "release_protected_advisory_caps": RELEASE_PROTECTED_ADVISORY_CAPS,
    "behavioral_cloning": BEHAVIORAL_CLONING,
    "mpc_acceptance_fallback": dict(CTRL.get("mpc_acceptance_fallback", {})),
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
}

cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
runtime_ctx = {
    "system": cstr,
    "agent": matrix_agent,
    "MPC_obj": MPC_obj,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": POLYMER_OBSERVER_POLES.copy(),
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "system_metadata": POLYMER_SYSTEM_METADATA,
    "reward_params": reward_params,
}

result_bundle = run_matrix_multiplier_supervisor(matrix_cfg=matrix_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = BASELINE_MPC_PATH

Sub_Episode: 1 | avg. reward: -3.3664894050756153 | alpha: 1.0 | delta: [1. 1.]
Sub_Episode: 2 | avg. reward: -4.417484074722275 | alpha: 1.0 | delta: [1. 1.]
Sub_Episode: 3 | avg. reward: -4.417713149544652 | alpha: 1.0 | delta: [1. 1.]
Sub_Episode: 4 | avg. reward: -4.417401159608377 | alpha: 1.0 | delta: [1. 1.]
Sub_Episode: 5 | avg. reward: -4.4172623108317115 | alpha: 1.0 | delta: [1. 1.]
Sub_Episode: 6 | avg. reward: -4.41737466725863 | alpha: 1.0 | delta: [1. 1.]
Sub_Episode: 7 | avg. reward: -4.417529762273493 | alpha: 1.0 | delta: [1. 1.]
Sub_Episode: 8 | avg. reward: -4.417120576866876 | alpha: 1.0 | delta: [1. 1.]
Sub_Episode: 9 | avg. reward: -4.41743528690505 | alpha: 1.0 | delta: [1. 1.]
Sub_Episode: 10 | avg. reward: -4.4174166259101435 | alpha: 1.0 | delta: [1. 1.]
Sub_Episode: 11 | avg. reward: -4.41743064324858 | alpha: 1.0 | delta: [1. 1.]
Sub_Episode: 12 | avg. reward: -4.417183720704538 | alpha: 1.0 | delta: [1. 1.]
Sub_Episode: 13 | avg. reward: -4.417326777946571

## Plotting And Export

In [8]:
# Generate saved figures and any requested MPC comparison plots.
out_dir_rl = plot_matrix_multiplier_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "save_pdf": SAVE_PDF,
    },
)

out_dir_cmp = compare_mpc_rl_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=BASELINE_MPC_PATH,
    reward_fn=reward_fn,
    directory=RESULT_DIR,
    prefix_name=COMPARE_PREFIX,
    compare_mode=RUN_PROFILE["compare_mode"],
    start_episode=COMPARE_START_EPISODE,
)

print("RL result directory:", out_dir_rl)
print("Comparison directory:", out_dir_cmp)


RL result directory: C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Polymer\Results\td3_multipliers_disturb\20260425_005143
Comparison directory: C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Polymer\Results\disturb_compare_td3_multipliers\20260425_005156
